# Experiment 21: Cross-Validation Evaluation

This experiment checks whether my previous single 80/20 validation split was giving mw an overly optimistic estimate. Instead of relying on one validation split, imma use stratified 5-fold cross-validation and evaluate several of my strongest model families using the same folds.

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42


## 2. Load Data

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('\nTarget distribution:')
print(train['Survived'].value_counts(normalize=True).sort_index())


Train shape: (891, 12)
Test shape: (418, 11)

Target distribution:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


## 3. Feature Engineering

In [3]:
def feature_engineering(df):
    df = df.copy()

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    df['AgeMissing'] = df['Age'].isna().astype(int)
    df['FareMissing'] = df['Fare'].isna().astype(int)
    df['EmbarkedMissing'] = df['Embarked'].isna().astype(int)

    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['CabinDeck'] = df['Cabin'].fillna('Unknown').astype(str).str[0]

    df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    df.loc[~df['Title'].isin(common_titles), 'Title'] = 'Rare'

    df['Surname'] = df['Name'].str.split(',').str[0].str.strip()

    df['TicketPrefix'] = (
        df['Ticket'].astype(str)
        .str.replace(r'\d', '', regex=True)
        .str.replace(r'[./ ]', '', regex=True)
        .replace('', 'NONE')
    )

    df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['SurnameGroupSize'] = df.groupby('Surname')['Surname'].transform('count')

    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].replace(0, np.nan)
    df['FarePerPersonMissing'] = df['FarePerPerson'].isna().astype(int)

    df['SexPclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)

    df['FamilySizeBand'] = pd.cut(
        df['FamilySize'],
        bins=[0, 1, 4, 7, np.inf],
        labels=['Alone', 'Small', 'Medium', 'Large']
    ).astype(str)

    df['AgeBand'] = pd.cut(
        df['Age'],
        bins=[-np.inf, 12, 18, 30, 50, np.inf],
        labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
    ).astype(str)

    df['FareBand'] = pd.qcut(
        df['Fare'],
        q=5,
        labels=['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh'],
        duplicates='drop'
    ).astype(str)

    df['FamilySex'] = df['Sex'].astype(str) + '_' + df['FamilySizeBand'].astype(str)
    df['PclassTitle'] = df['Pclass'].astype(str) + '_' + df['Title'].astype(str)
    df['PclassAgeBand'] = df['Pclass'].astype(str) + '_' + df['AgeBand'].astype(str)
    df['FamilyTicket'] = df['FamilySize'].astype(str) + '_' + df['TicketPrefix'].astype(str)

    df['NameLength'] = df['Name'].astype(str).str.len()
    df['NameWords'] = df['Name'].astype(str).str.split().str.len()
    df['TicketLength'] = df['Ticket'].astype(str).str.len()
    df['CabinCount'] = df['Cabin'].fillna('').astype(str).str.split().str.len()
    df['DeckKnown'] = (df['Cabin'].notna()).astype(int)

    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)
    df['SmallFamily'] = ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype(int)
    df['FemaleChild'] = (
        (df['Sex'] == 'female') &
        (df['Age'].fillna(-1) <= 12)
    ).astype(int)

    df['FarePerAge'] = df['Fare'] / df['Age'].replace(0, np.nan)
    df['ClassFare'] = df['Pclass'] * df['Fare']
    df['SiblingChildRatio'] = df['SibSp'] / (df['Parch'] + 1)
    df['FamilyFare'] = df['Fare'] / df['FamilySize'].replace(0, np.nan)
    df['SexTitle'] = df['Sex'].astype(str) + '_' + df['Title'].astype(str)

    return df


X = feature_engineering(train.drop(columns=['Survived']))
y = train['Survived'].copy()

X = X.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname'])

print('Engineered feature shape:', X.shape)


Engineered feature shape: (891, 41)


## 4. Preprocessing

In [4]:
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


## 5. Cross-Validation Setup

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

folds = list(cv.split(X, y))

print('Number of folds:', len(folds))
for i, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(f'Fold {i}: train={len(train_idx)}, validation={len(valid_idx)}')


Number of folds: 5
Fold 1: train=712, validation=179
Fold 2: train=713, validation=178
Fold 3: train=713, validation=178
Fold 4: train=713, validation=178
Fold 5: train=713, validation=178


## 6. Model Evaluation

In [6]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=900,
        max_depth=3,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=4,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    'CatBoost': CatBoostClassifier(
        iterations=600,
        depth=5,
        learning_rate=0.03,
        loss_function='Logloss',
        verbose=False,
        random_seed=RANDOM_STATE,
        l2_leaf_reg=5
    )
}


def build_pipeline(model):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])


cv_results = []

for model_name, model in models.items():
    fold_scores = []

    print(f'\n{model_name}')
    print('-' * len(model_name))

    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        pipeline = build_pipeline(model)
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_valid)

        score = accuracy_score(y_valid, predictions)
        fold_scores.append(score)

        print(f'Fold {fold_number}: {score:.4f}')

    cv_results.append({
        'Model': model_name,
        'Fold 1': fold_scores[0],
        'Fold 2': fold_scores[1],
        'Fold 3': fold_scores[2],
        'Fold 4': fold_scores[3],
        'Fold 5': fold_scores[4],
        'Mean': np.mean(fold_scores),
        'Std': np.std(fold_scores, ddof=1),
        'Min': np.min(fold_scores),
        'Max': np.max(fold_scores)
    })



Logistic Regression
-------------------
Fold 1: 0.8492
Fold 2: 0.8258
Fold 3: 0.7921
Fold 4: 0.8315
Fold 5: 0.8315

Random Forest
-------------
Fold 1: 0.8436
Fold 2: 0.8258
Fold 3: 0.8315
Fold 4: 0.8371
Fold 5: 0.8427

XGBoost
-------
Fold 1: 0.8659
Fold 2: 0.8315
Fold 3: 0.8034
Fold 4: 0.8371
Fold 5: 0.8371

CatBoost
--------
Fold 1: 0.8603
Fold 2: 0.8539
Fold 3: 0.8146
Fold 4: 0.8427
Fold 5: 0.8371


## 7. Cross-Validation Results

In [7]:
results_df = pd.DataFrame(cv_results).sort_values(
    'Mean',
    ascending=False
).reset_index(drop=True)

print('\n5-Fold Cross-Validation Results')
display(results_df.round(4))



5-Fold Cross-Validation Results


,Model,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std,Min,Max
0,CatBoost,0.8603,0.8539,0.8146,0.8427,0.8371,0.8417,0.0177,0.8146,0.8603
1,Random Forest,0.8436,0.8258,0.8315,0.8371,0.8427,0.8361,0.0075,0.8258,0.8436
2,XGBoost,0.8659,0.8315,0.8034,0.8371,0.8371,0.8350,0.0222,0.8034,0.8659
3,Logistic Regression,0.8492,0.8258,0.7921,0.8315,0.8315,0.8260,0.0209,0.7921,0.8492


## 8. Old Validation vs Cross-Validation

In [8]:
X_train_old, X_valid_old, y_train_old, y_valid_old = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

old_results = []

for model_name, model in models.items():
    pipeline = build_pipeline(model)
    pipeline.fit(X_train_old, y_train_old)
    predictions = pipeline.predict(X_valid_old)
    old_score = accuracy_score(y_valid_old, predictions)

    cv_mean = results_df.loc[
        results_df['Model'] == model_name,
        'Mean'
    ].iloc[0]

    old_results.append({
        'Model': model_name,
        'Old 80/20 Accuracy': old_score,
        '5-Fold Mean': cv_mean,
        'Difference': old_score - cv_mean
    })

old_vs_cv_df = pd.DataFrame(old_results).sort_values(
    '5-Fold Mean',
    ascending=False
).reset_index(drop=True)

print('Old 80/20 Validation vs 5-Fold Cross-Validation')
display(old_vs_cv_df.round(4))


Old 80/20 Validation vs 5-Fold Cross-Validation


,Model,Old 80/20 Accuracy,5-Fold Mean,Difference
0,CatBoost,0.8156,0.8417,-0.0261
1,Random Forest,0.8324,0.8361,-0.0037
2,XGBoost,0.8156,0.8350,-0.0193
3,Logistic Regression,0.8268,0.8260,0.0008


## 9. Conclusion

In [9]:
best_row = results_df.iloc[0]

print('Experiment 21 Summary')
print('=' * 60)
print(f"Highest 5-fold mean: {best_row['Model']}")
print(f"Mean CV accuracy:    {best_row['Mean']:.4f}")
print(f"CV standard dev:     {best_row['Std']:.4f}")
print(f"Minimum fold:        {best_row['Min']:.4f}")
print(f"Maximum fold:        {best_row['Max']:.4f}")

old_best = old_vs_cv_df.loc[
    old_vs_cv_df['Model'] == best_row['Model']
        ,
        'Old 80/20 Accuracy'
        ].iloc[0]

print(f"Old 80/20 accuracy:   {old_best:.4f}")
print(f"Old vs CV difference: {old_best - best_row['Mean']:+.4f}")

if old_best > best_row['Mean']:
    print('\nThe old single split was more optimistic than the 5-fold estimate.')
elif old_best < best_row['Mean']:
    print('\nThe old single split was more pessimistic than the 5-fold estimate.')
else:
    print('\nThe old single split was almost identical to the 5-fold estimate.')

print('\nUse the 5-fold mean as the main validation reference for future experiments.')


Experiment 21 Summary
Highest 5-fold mean: CatBoost
Mean CV accuracy:    0.8417
CV standard dev:     0.0177
Minimum fold:        0.8146
Maximum fold:        0.8603
Old 80/20 accuracy:   0.8156
Old vs CV difference: -0.0261

The old single split was more pessimistic than the 5-fold estimate.

Use the 5-fold mean as the main validation reference for future experiments.
